In [1]:
import os, json
from pathlib import Path

BASE    = r'C:\Users\varma\OneDrive\Desktop\newml'
FOLDERS = [
    os.path.join(BASE, 'models'),
    os.path.join(BASE, 'logs'),
    os.path.join(BASE, 'reports'),
    os.path.join(BASE, 'graphs'),
    os.path.join(BASE, 'quarantine'),
    os.path.join(BASE, 'yara_rules'),
    os.path.join(BASE, 'shap_plots'),
]
for f in FOLDERS:
    Path(f).mkdir(parents=True, exist_ok=True)

def load_env(path):
    if not os.path.exists(path): return
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            if '=' not in line: continue
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()

load_env(os.path.join(BASE, '.env'))
print('Folders verified. .env loaded.')

Folders verified. .env loaded.


In [2]:
!pip install shap
import os, sys, json, joblib, warnings, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import psutil, threading
from datetime import datetime
from collections import Counter, deque
warnings.filterwarnings('ignore')

try:
    import shap
    HAS_SHAP = True
    print(f'SHAP {shap.__version__} ready')
except ImportError:
    HAS_SHAP = False
    print('SHAP not installed — run: pip install shap')

try:
    import networkx as nx
    HAS_NX = True
    print('NetworkX ready')
except ImportError:
    HAS_NX = False
    print('NetworkX not installed — run: pip install networkx')

try:
    import tensorflow as tf
    HAS_TF = True
    print(f'TF {tf.__version__} ready')
except ImportError:
    HAS_TF = False

PAL = ['#4f7cff','#22c55e','#f59e0b','#ef4444','#8b5cf6','#06b6d4','#ec4899','#14b8a6']
SEV_COL = {
    'CLEAN':'#22c55e','LOW':'#06b6d4',
    'MEDIUM':'#f59e0b','HIGH':'#f97316','CRITICAL':'#ef4444'
}
RANDOM = 42
np.random.seed(RANDOM)
print('All imports ready.')

SHAP 0.51.0 ready
NetworkX ready
TF 2.21.0 ready
All imports ready.


In [3]:
!pip install tensorflow
BASE     = r'C:\Users\varma\OneDrive\Desktop\newml'
MODELS   = os.path.join(BASE, 'models')
NB1_MDL  = os.path.join(BASE, 'ml')
NB2_MDL  = os.path.join(BASE, 'dl')
NB3_MDL  = os.path.join(BASE, 'advanced')
LOG_FILE = os.path.join(BASE, 'logs', 'protector_v2.log')

def load_env(path):
    if not os.path.exists(path): return
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            if '=' not in line: continue
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()

load_env(os.path.join(BASE, '.env'))

GROQ_KEY     = os.environ.get('GROQ_KEY', '')
GROQ_MODEL   = os.environ.get('GROQ_MODEL', 'llama-3.3-70b-versatile')
ALERT_EMAIL  = os.environ.get('ALERT_EMAIL', '')
ALERT_THRESH = float(os.environ.get('THREAT_ALERT_THRESHOLD', '0.5'))
CRIT_THRESH  = float(os.environ.get('CRITICAL_ALERT_THRESHOLD', '0.7'))
ENABLE_DL    = os.environ.get('ENABLE_DL_MODELS', 'true').lower() == 'true'

def safe_load(fname):
    for folder in [MODELS, NB1_MDL, NB2_MDL, NB3_MDL]:
        path = os.path.join(folder, fname)
        if not os.path.exists(path): continue
        try:
            if fname.endswith('.pkl'):
                m = joblib.load(path)
            elif fname.endswith('.json'):
                with open(path, encoding='utf-8') as f:
                    m = json.load(f)
            elif fname.endswith('.txt'):
                with open(path, encoding='utf-8') as f:
                    s = set(f.read().splitlines())
                    s.discard('')
                    return s
            elif fname.endswith('.npy'):
                m = np.load(path, allow_pickle=True)
            else:
                m = path
            sz = os.path.getsize(path)
            src = ('ml'  if NB1_MDL in path else
                   'dl'  if NB2_MDL in path else
                   'adv' if NB3_MDL in path else 'models')
            print(f'  OK [{src:>6}] {fname:<45} ({sz:>10,} bytes)')
            return m
        except Exception as e:
            print(f'  ERR {fname}: {e}')
    print(f'  MISSING: {fname}')
    return None

print('='*60)
print('  Protector v2 — Loading Models')
print('='*60)

# NB1 — Classical
print('\n[NB1 Classical]')
SCALER     = safe_load('v2_scaler.pkl')
IMPUTER    = safe_load('v2_imputer.pkl')
FEAT_NAMES = safe_load('v2_feature_names.json')
CLASSES    = safe_load('v2_label_classes.json')
OCSVM      = safe_load('v2_ocsvm.pkl')
ISO_FOREST = safe_load('v2_isoforest.pkl')
SVM_BIN    = safe_load('v2_svm.pkl')
DT_MULTI   = safe_load('v2_dt.pkl')
RF_MULTI   = safe_load('v2_rf.pkl')
GNB_MULTI  = safe_load('v2_gnb.pkl')
ELM_MULTI  = safe_load('v2_elm.pkl')
GBC_ENS    = safe_load('v2_gbc.pkl')
ADA_ENS    = safe_load('v2_ada.pkl')
BAG_ENS    = safe_load('v2_bagging.pkl')
VOTING_ENS = safe_load('v2_voting.pkl')

# NB2 — Deep Learning
print('\n[NB2 Deep Learning]')
HAS_DL = False
DNN_MDL = AE_MDL = SEQ_SC = None
AE_THRESH = 0.01
NB2_CLASSES = CLASSES

if ENABLE_DL:
    try:
        import tensorflow as tf
        def load_keras(fname):
            for folder in [MODELS, NB2_MDL]:
                p = os.path.join(folder, fname)
                if os.path.exists(p):
                    return tf.keras.models.load_model(p)
            return None
        DNN_MDL     = load_keras('v2_dnn.keras')
        AE_MDL      = load_keras('v2_autoencoder.keras')
        ae_t        = safe_load('v2_ae_threshold.json')
        AE_THRESH   = ae_t.get('threshold', 0.01) if ae_t else 0.01
        SEQ_SC      = safe_load('v2_scaler.pkl')
        NB2_CLASSES = safe_load('v2_label_classes.json') or CLASSES
        HAS_DL      = True
        print(f'  DL models loaded. AE_THRESH={AE_THRESH}')
    except Exception as e:
        print(f'  DL unavailable: {e}')

# NB3 — Advanced
print('\n[NB3 Advanced]')
HMM_MODELS = safe_load('v2_hmm_models.pkl')
KMEANS     = safe_load('v2_kmeans.pkl')
PCA2VEC    = safe_load('v2_pca2vec.pkl')
HOST_SCALER= safe_load('v2_host_scaler.pkl')

# NB5 — Explainability + Family
print('\n[NB5 Explainability]')
FAMILY_CLF   = safe_load('v2_family_clf.pkl')
FAMILY_ENC   = safe_load('v2_family_encoder.pkl')
FAMILY_IMP   = safe_load('v2_family_imputer.pkl')
FAMILY_NAMES = safe_load('v2_family_names.json')
META_CLF     = safe_load('v2_ensemble_meta.pkl')
META_ORDER   = safe_load('v2_ensemble_order.json')
SHAP_EXPLN   = safe_load('v2_shap_explainer.pkl') if HAS_SHAP else None

# Threat intel
print('\n[Threat Intel]')
BAD_HASHES   = safe_load('bad_hashes.txt')   or set()
THREAT_INTEL = safe_load('threat_intel.json') or {}
MALICIOUS_IPS= set(THREAT_INTEL.get('malicious_ips', []))

print('\n' + '='*60)
print(f'  SCALER     : {"OK" if SCALER else "MISSING"}')
print(f'  RF_MULTI   : {"OK" if RF_MULTI else "MISSING"}')
print(f'  FAMILY_CLF : {"OK" if FAMILY_CLF else "MISSING (run NB5)"}')
print(f'  SHAP       : {"OK" if SHAP_EXPLN else "MISSING (run NB5)"}')
print(f'  DL models  : {"OK" if HAS_DL else "disabled"}')
print(f'  GROQ_KEY   : {"SET" if GROQ_KEY and GROQ_KEY != "your_groq_api_key_here" else "NOT SET"}')
print('='*60)

  Protector v2 — Loading Models

[NB1 Classical]
  OK [models] v2_scaler.pkl                                 (     1,751 bytes)
  OK [models] v2_imputer.pkl                                (     2,583 bytes)
  OK [models] v2_feature_names.json                         (     1,304 bytes)
  OK [models] v2_label_classes.json                         (        46 bytes)
  OK [models] v2_ocsvm.pkl                                  (   647,359 bytes)
  OK [models] v2_isoforest.pkl                              (13,380,633 bytes)
  OK [models] v2_svm.pkl                                    ( 4,792,452 bytes)
  OK [models] v2_dt.pkl                                     (     3,345 bytes)
  OK [models] v2_rf.pkl                                     (   303,457 bytes)
  OK [models] v2_gnb.pkl                                    (     4,271 bytes)
  ERR v2_elm.pkl: Can't get attribute 'ELMClassifier' on <module '__main__'>
  ERR v2_elm.pkl: Can't get attribute 'ELMClassifier' on <module '__main__'>
  MISSI

In [4]:
HISTORY      = deque(maxlen=200)
alert_log    = []
scan_history = []

_LIVE_MEANS = np.array([10,30,50,500,1e8,2e8,0,50,2e6,0.5,0,8000], dtype=np.float32)
_LIVE_STDS  = np.array([15,20,80,300,3e8,5e8,1,100,5e6,0.3,0.5,20000], dtype=np.float32)

def scale_live(f):
    return ((f - _LIVE_MEANS) / (_LIVE_STDS + 1e-9)).reshape(1, -1)

def pad_to_78(f):
    v = np.zeros(78, dtype=np.float32)
    v[0]=f[11]; v[2]=f[4]+f[5]; v[3]=f[2]; v[4]=f[2]
    v[5]=f[4];  v[6]=f[5]
    v[14]=(f[4]+f[5])/max(1.0,f[2])
    v[15]=f[2]; v[41]=f[8]; v[53]=f[8]; v[75]=f[9]
    return v.reshape(1, -1)

def scale_78(f):
    v78 = pad_to_78(f)
    if SCALER:
        try: return SCALER.transform(v78)
        except: pass
    return v78

def extract_features():
    try:
        net   = psutil.net_io_counters()
        bsent = float(net.bytes_sent)
        brecv = float(net.bytes_recv)
        disk  = float(psutil.disk_io_counters().read_bytes +
                      psutil.disk_io_counters().write_bytes) / 1e9
    except:
        bsent = brecv = disk = 0.0
    cpu   = float(psutil.cpu_percent(interval=0.1))
    mem   = float(psutil.virtual_memory().percent)
    conns = float(len(psutil.net_connections(kind='inet')))
    thrs  = 0.0
    try:
        thrs = float(sum(p.num_threads()
                         for p in psutil.process_iter(['num_threads'])
                         if p.info['num_threads']))
    except: pass
    feat = np.array([cpu, mem, conns, thrs, bsent, brecv,
                     0.0, disk, bsent/max(1,conns),
                     min(1.0,conns/100), 0.0, 80.0],
                    dtype=np.float32)
    HISTORY.append({'time':datetime.now(),'feat':feat,
                    'cpu':cpu,'mem':mem,'conns':conns})
    return feat

def get_process_snapshot():
    procs = []
    for p in psutil.process_iter(['pid','name','cpu_percent',
                                   'memory_percent','connections',
                                   'num_threads','username','exe']):
        try:
            i = p.info
            procs.append({
                'pid'  : i['pid'],
                'name' : (i['name'] or 'unknown')[:20],
                'cpu'  : round(i['cpu_percent']    or 0, 2),
                'mem'  : round(i['memory_percent'] or 0, 3),
                'conns': len(i['connections'] or []),
                'thrs' : i['num_threads'] or 1,
                'user' : i['username'] or '',
                'exe'  : i['exe'] or '',
            })
        except: pass
    return sorted(procs, key=lambda x: x['cpu'], reverse=True)

def get_network_snapshot():
    conns = []
    for c in psutil.net_connections(kind='inet'):
        if not c.raddr: continue
        conns.append({
            'rip'      : c.raddr.ip,
            'rport'    : c.raddr.port,
            'status'   : c.status,
            'pid'      : c.pid or 0,
            'malicious': c.raddr.ip in MALICIOUS_IPS,
        })
    return conns

feat_test = extract_features()
print(f'Feature extractor OK: shape={feat_test.shape}')
print(f'scale_live : {scale_live(feat_test).shape}')
print(f'pad_to_78  : {pad_to_78(feat_test).shape}')
print(f'scale_78   : {scale_78(feat_test).shape}')

Feature extractor OK: shape=(12,)
scale_live : (1, 12)
pad_to_78  : (1, 78)
scale_78   : (1, 78)


In [5]:
class AgentResult:
    def __init__(self, name, verdict, confidence, detail='', proba=None):
        self.name       = name
        self.verdict    = verdict
        self.confidence = confidence
        self.detail     = detail
        self.proba      = proba  # full probability vector
        self.is_threat  = verdict not in ('normal','NORMAL','Benign',
                                           'Normal','cluster_0',0)

def run_all_agents(feat_raw):
    results = []
    Xs12 = scale_live(feat_raw)
    X78  = pad_to_78(feat_raw)
    Xs78 = scale_78(feat_raw)

    # Anomaly agents
    for model, name, inp, kind in [
        (OCSVM,      'OC-SVM',    Xs12, 'anomaly'),
        (ISO_FOREST, 'IsoForest', Xs12, 'anomaly'),
        (SVM_BIN,    'SVM-Bin',   Xs12, 'binary'),
    ]:
        if model is None: continue
        try:
            if kind == 'anomaly':
                pred  = model.predict(inp)[0]
                score = model.decision_function(inp)[0]
                conf  = max(0.0, min(1.0, (score+1)/2))
                results.append(AgentResult(name,
                    'normal' if pred==1 else 'ANOMALY',
                    1-conf if pred==-1 else conf))
            else:
                pred  = model.predict(inp)[0]
                proba = model.predict_proba(inp)[0]
                results.append(AgentResult(name,
                    'ATTACK' if pred==1 else 'normal',
                    float(proba.max()), proba=proba))
        except: pass

    # Multi-class agents
    for name, model, use_sc in [
        ('DT',        DT_MULTI,   False),
        ('RF',        RF_MULTI,   False),
        ('GNB',       GNB_MULTI,  True),
        ('ELM',       ELM_MULTI,  True),
        ('GradBoost', GBC_ENS,    False),
        ('AdaBoost',  ADA_ENS,    False),
        ('Bagging',   BAG_ENS,    False),
        ('Voting',    VOTING_ENS, True),
    ]:
        if model is None or CLASSES is None: continue
        try:
            inp   = Xs78 if use_sc else X78
            pred  = model.predict(inp)[0]
            proba = model.predict_proba(inp)[0]
            cls   = CLASSES[pred] if pred < len(CLASSES) else 'unknown'
            results.append(AgentResult(name, cls,
                float(proba.max()), proba=proba))
        except: pass

    # Ensemble meta-learner
    if META_CLF and META_ORDER and CLASSES:
        try:
            meta_probas = []
            for mname in META_ORDER:
                model_map = {
                    'RF':RF_MULTI,'DT':DT_MULTI,'GNB':GNB_MULTI,
                    'ELM':ELM_MULTI,'GBC':GBC_ENS
                }
                m = model_map.get(mname)
                if m is None: continue
                inp = Xs78 if mname in ('GNB','ELM') else X78
                meta_probas.append(m.predict_proba(inp)[0])
            if meta_probas:
                meta_feat  = np.hstack(meta_probas).reshape(1,-1)
                meta_pred  = META_CLF.predict(meta_feat)[0]
                meta_proba = META_CLF.predict_proba(meta_feat)[0]
                cls        = CLASSES[meta_pred] if meta_pred < len(CLASSES) else 'unknown'
                results.append(AgentResult('MetaEnsemble', cls,
                    float(meta_proba.max()),
                    detail='stacked meta-learner',
                    proba=meta_proba))
        except Exception as e:
            pass

    # Family classifier
    if FAMILY_CLF and FAMILY_NAMES:
        try:
            pred_f  = FAMILY_CLF.predict(X78)[0]
            proba_f = FAMILY_CLF.predict_proba(X78)[0]
            family  = FAMILY_NAMES[pred_f] if pred_f < len(FAMILY_NAMES) else 'unknown'
            is_thr  = family not in ('Normal',)
            results.append(AgentResult('FamilyCLF', family,
                float(proba_f.max()),
                detail=f'attack family: {family}',
                proba=proba_f))
        except: pass

    # KMeans
    if KMEANS:
        try:
            c = KMEANS.predict(Xs12)[0]
            d = KMEANS.transform(Xs12)[0].min()
            results.append(AgentResult('KMeans', f'cluster_{c}',
                max(0.0, 1.0-d/10.0), f'dist={d:.2f}'))
        except: pass

    # HMM
    if HMM_MODELS:
        try:
            chunk = np.tile(Xs12[0], (20,1))
            scores = {}
            for cls, m in HMM_MODELS.items():
                try:    scores[cls] = m.score(chunk)/20
                except: scores[cls] = -1e9
            best = max(scores, key=scores.get)
            conf = min(1.0, max(0.0, (scores[best]+50)/100))
            results.append(AgentResult('HMM', best, conf,
                f'log-L={scores[best]:.2f}'))
        except: pass

    # DNN
    if HAS_DL and DNN_MDL and NB2_CLASSES:
        try:
            proba = DNN_MDL.predict(Xs78, verbose=0)[0]
            pred  = int(np.argmax(proba))
            cls   = NB2_CLASSES[pred] if pred < len(NB2_CLASSES) else 'unknown'
            results.append(AgentResult('DNN', cls,
                float(proba.max()), proba=proba))
        except: pass

    # Autoencoder
    if HAS_DL and AE_MDL:
        try:
            T    = 20
            ae_f = AE_MDL.input_shape[-1]
            base = Xs12 if ae_f==12 else Xs78
            seq  = np.tile(base, (T,1)).reshape(1,T,-1)
            recon= AE_MDL.predict(seq, verbose=0)
            err  = float(np.mean((recon-seq)**2))
            is_a = err > AE_THRESH
            results.append(AgentResult('Autoencoder',
                'ANOMALY' if is_a else 'normal',
                min(1.0,err/AE_THRESH) if is_a else max(0.0,1.0-err/AE_THRESH),
                f'recon_err={err:.5f}'))
        except: pass

    return results

def fuse_agents(results):
    if not results: return 'unknown', 0.0, {}
    threat_agents = [r for r in results if r.is_threat]
    threat_score  = sum(r.confidence for r in threat_agents) / max(len(results),1)
    non_normal    = [r.verdict for r in threat_agents
                     if r.verdict not in ('normal','NORMAL','ANOMALY',
                                           'Benign','Normal','cluster_0')]
    attack_type   = (Counter(non_normal).most_common(1)[0][0]
                     if non_normal else
                     ('ANOMALY' if threat_agents else 'normal'))
    family_votes  = [r.verdict for r in results if r.name == 'FamilyCLF']
    attack_family = family_votes[0] if family_votes else 'Unknown'

    if   threat_score > 0.7: sev = 'CRITICAL'
    elif threat_score > 0.5: sev = 'HIGH'
    elif threat_score > 0.3: sev = 'MEDIUM'
    elif threat_score > 0.1: sev = 'LOW'
    else:                    sev = 'CLEAN'

    return attack_type, threat_score, {
        'verdict'      : attack_type if threat_score > 0.2 else 'normal',
        'severity'     : sev,
        'threat_score' : round(threat_score, 3),
        'attack_family': attack_family,
        'n_agents'     : len(results),
        'n_flagging'   : len(threat_agents),
        'agents'       : {r.name: {
                            'verdict': r.verdict,
                            'conf'   : round(r.confidence, 3),
                            'detail' : r.detail,
                          } for r in results}
    }

# Test
feat = extract_features()
res  = run_all_agents(feat)
v, s, summ = fuse_agents(res)
print(f'Verdict      : {summ["verdict"]}')
print(f'Severity     : {summ["severity"]}')
print(f'Attack Family: {summ["attack_family"]}')
print(f'Threat Score : {s:.3f}')
print(f'Agents       : {summ["n_agents"]} | Flagging: {summ["n_flagging"]}')
for name, info in summ['agents'].items():
    flag = '!!' if info['verdict'] not in ('normal','NORMAL','Benign',
                                            'Normal','cluster_0') else '  '
    print(f'  {flag} {name:<16} {info["verdict"]:<22} conf={info["conf"]:.3f}')

Verdict      : normal
Severity     : LOW
Attack Family: Unknown
Threat Score : 0.163
Agents       : 3 | Flagging: 1
     AdaBoost         Benign                 conf=0.511
  !! Voting           SSH-Bruteforce         conf=0.488
     HMM              normal                 conf=0.000


In [6]:
def explain_with_shap(feat_raw, top_n=12):
    if not HAS_SHAP or SHAP_EXPLN is None:
        print('SHAP not available. Run NB5 first.')
        return {}

    # Load feature alignment meta
    shap_meta = safe_load('v2_shap_meta.json') or {}
    feat_cols = shap_meta.get('feature_cols', FEAT_NAMES or [])
    n_feat    = len(feat_cols)

    X78 = pad_to_78(feat_raw)[:, :n_feat]   # align to SHAP training features

    try:
        shap_vals = SHAP_EXPLN.shap_values(X78)

        # Normalise to array
        if isinstance(shap_vals, list):
            shap_arr = np.stack(shap_vals, axis=0)   # (n_cls, 1, n_feat)
        elif isinstance(shap_vals, np.ndarray):
            if shap_vals.ndim == 2:
                shap_arr = shap_vals[np.newaxis, :, :]
            else:
                shap_arr = shap_vals
        else:
            shap_arr = np.array(shap_vals)

        # Mean abs across classes, squeeze sample dim
        mean_abs = np.abs(shap_arr).mean(axis=0).squeeze()   # (n_feat,)
        n_common = min(len(mean_abs), len(feat_cols))
        mean_abs = mean_abs[:n_common]
        f_cols   = feat_cols[:n_common]

        fi_dict = dict(sorted(
            zip(f_cols, mean_abs.tolist()),
            key=lambda x: abs(x[1]), reverse=True))

        top_feats = list(fi_dict.keys())[:top_n]
        top_vals  = [fi_dict[k] for k in top_feats]

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle('SHAP Explainability — Live Prediction',
                      fontsize=13, fontweight='bold')

        colors = ['#ef4444' if v > 0 else '#4f7cff' for v in top_vals[::-1]]
        axes[0].barh(top_feats[::-1], top_vals[::-1],
                     color=colors, edgecolor='white')
        axes[0].axvline(0, color='black', linewidth=0.8)
        axes[0].set_title(f'Top {top_n} SHAP Values')
        axes[0].set_xlabel('SHAP value')

        cumulative = np.cumsum(top_vals)
        axes[1].barh(top_feats[::-1], cumulative[::-1],
                     color=SEV_COL.get('HIGH','#f97316'), alpha=0.7)
        axes[1].set_title('SHAP Cumulative Impact')
        axes[1].set_xlabel('Cumulative SHAP')

        plt.tight_layout()
        save_p = os.path.join(BASE,'shap_plots',
            f'shap_live_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png')
        os.makedirs(os.path.dirname(save_p), exist_ok=True)
        plt.savefig(save_p, dpi=120, bbox_inches='tight')
        plt.show()
        write_log(f'SHAP plot saved: {save_p}')
        return fi_dict

    except Exception as e:
        print(f'SHAP explain error: {e}')
        import traceback; traceback.print_exc()
        return {}

print('SHAP explain function updated.')

SHAP explain function updated.


In [7]:
import schedule

monitor_active = True
STATUS = {'verdict':'Starting','severity':'UNKNOWN',
          'score':0.0,'summ':{},'feat':None}

def write_log(msg, level='INFO'):
    ts   = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{ts}] [{level}] {msg}'
    try:
        with open(LOG_FILE, 'a', encoding='utf-8') as f:
            f.write(line + '\n')
    except: pass
    print(line)

def send_notification(title, msg):
    try:
        from plyer import notification
        notification.notify(title=title, message=msg[:250],
                            app_name='Protector v2', timeout=10)
    except: pass

def send_email(subject, body):
    try:
        import smtplib
        from email.mime.text import MIMEText
        msg = MIMEText(body)
        msg['Subject'] = subject
        msg['From'] = msg['To'] = ALERT_EMAIL
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(ALERT_EMAIL, os.environ.get('EMAIL_PASS',''))
            smtp.send_message(msg)
        write_log(f'Email sent: {subject}')
    except Exception as e:
        write_log(f'Email failed: {e}', 'WARN')

def monitor_cycle():
    feat = extract_features()
    res  = run_all_agents(feat)
    v, s, summ = fuse_agents(res)
    STATUS.update({'verdict':v,'severity':summ['severity'],
                   'score':s,'summ':summ,'feat':feat})
    entry = {
        'time'         : datetime.now().isoformat(),
        'verdict'      : summ['verdict'],
        'severity'     : summ['severity'],
        'threat_score' : s,
        'attack_family': summ['attack_family'],
        'n_flagging'   : summ['n_flagging'],
        'cpu'          : float(feat[0]),
        'mem'          : float(feat[1]),
        'conns'        : float(feat[2]),
    }
    scan_history.append(entry)
    if len(scan_history) > 500: scan_history.pop(0)

    if s > ALERT_THRESH:
        alert_log.append(entry)
        write_log(f'THREAT: {v} family={summ["attack_family"]}'
                  f' score={s:.3f} sev={summ["severity"]}', 'ALERT')
        send_notification(
            f'Protector v2 — {summ["severity"]}',
            f'{v} ({summ["attack_family"]}) | score={s:.2f}'
            f' | {summ["n_flagging"]}/{summ["n_agents"]} agents')
        if s > CRIT_THRESH:
            body = (f'Score: {s:.3f}\nSeverity: {summ["severity"]}\n'
                    f'Attack family: {summ["attack_family"]}\n'
                    f'Time: {entry["time"]}\n\nAgents:\n'
                    + '\n'.join(f'  {k}: {vv["verdict"]} ({vv["conf"]:.2f})'
                                for k,vv in summ['agents'].items()))
            send_email(f'[Protector v2] CRITICAL: {v}', body)
    else:
        write_log(f'CLEAN score={s:.3f} family={summ["attack_family"]}')
    return entry, summ

schedule.every(60).seconds.do(monitor_cycle)

def scheduler_loop():
    while monitor_active:
        schedule.run_pending()
        time.sleep(1)

monitor_thread = threading.Thread(target=scheduler_loop, daemon=True)
monitor_thread.start()

entry, summ = monitor_cycle()
write_log('Protector v2 monitor started.')
print(f'Monitor running every 60s.')
print(f'First scan: {summ["verdict"]} | {summ["severity"]}'
      f' | family={summ["attack_family"]} | score={summ["threat_score"]}')

[2026-05-01 15:55:40] [INFO] CLEAN score=0.163 family=Unknown
[2026-05-01 15:55:40] [INFO] Protector v2 monitor started.
Monitor running every 60s.
First scan: normal | LOW | family=Unknown | score=0.163
[2026-05-01 15:56:41] [INFO] CLEAN score=0.163 family=Unknown
[2026-05-01 15:57:45] [INFO] CLEAN score=0.163 family=Unknown
[2026-05-01 15:58:49] [INFO] CLEAN score=0.163 family=Unknown
[2026-05-01 15:59:54] [INFO] CLEAN score=0.163 family=Unknown
[2026-05-01 16:01:19] [INFO] CLEAN score=0.163 family=Unknown
[2026-05-01 16:02:22] [INFO] CLEAN score=0.163 family=Unknown
[2026-05-01 16:03:26] [INFO] CLEAN score=0.163 family=Unknown
[2026-05-01 16:04:31] [INFO] CLEAN score=0.163 family=Unknown
[2026-05-01 16:05:35] [INFO] CLEAN score=0.164 family=Unknown


In [8]:
def plot_resource_dashboard(save=True):
    fig = plt.figure(figsize=(22,13))
    fig.suptitle(
        f'Protector v2 — Resource Dashboard  '
        f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}]   '
        f'Verdict: {STATUS["verdict"]}  |  '
        f'Severity: {STATUS["severity"]}  |  '
        f'Score: {STATUS["score"]:.3f}  |  '
        f'Family: {STATUS["summ"].get("attack_family","?")}',
        fontsize=12, fontweight='bold', y=0.99)
    gs = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.35)

    # CPU per core
    ax1 = fig.add_subplot(gs[0, :2])
    cpu_cores = psutil.cpu_percent(percpu=True)
    cols_cpu  = ['#ef4444' if c>80 else '#f59e0b' if c>50 else '#4f7cff'
                 for c in cpu_cores]
    bars = ax1.bar(range(len(cpu_cores)), cpu_cores,
                   color=cols_cpu, edgecolor='white')
    ax1.set_ylim(0, 100)
    ax1.axhline(80, color='#ef4444', linestyle='--', alpha=0.5)
    ax1.set_title(f'CPU Per Core  (avg={sum(cpu_cores)/len(cpu_cores):.1f}%)')
    ax1.set_xlabel('Core')
    for bar in bars:
        ax1.text(bar.get_x()+bar.get_width()/2,
                  bar.get_height()+1,
                  f'{bar.get_height():.0f}',
                  ha='center', fontsize=7)

    # RAM pie
    ax2 = fig.add_subplot(gs[0, 2])
    mem = psutil.virtual_memory()
    ax2.pie([mem.used, mem.available],
            labels=[f'Used\n{mem.used/1e9:.1f}GB',
                    f'Free\n{mem.available/1e9:.1f}GB'],
            colors=['#ef4444','#22c55e'],
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize':8})
    ax2.set_title(f'RAM ({mem.total/1e9:.1f}GB)')

    # Disk
    ax3 = fig.add_subplot(gs[0, 3])
    parts = psutil.disk_partitions()
    p_names, p_used, p_free = [], [], []
    for p in parts[:4]:
        try:
            u = psutil.disk_usage(p.mountpoint)
            p_names.append(p.mountpoint[:6])
            p_used.append(u.used/1e9)
            p_free.append(u.free/1e9)
        except: pass
    if p_names:
        x = np.arange(len(p_names))
        ax3.bar(x-0.2, p_used, 0.35, label='Used GB',
                color='#ef4444', alpha=0.85)
        ax3.bar(x+0.2, p_free, 0.35, label='Free GB',
                color='#22c55e', alpha=0.85)
        ax3.set_xticks(x); ax3.set_xticklabels(p_names)
        ax3.legend(fontsize=7)
    ax3.set_title('Disk Usage')

    # Top processes
    ax4 = fig.add_subplot(gs[1, :2])
    procs = get_process_snapshot()[:10]
    if procs:
        pn = [p['name'] for p in procs]
        pc = [p['cpu']  for p in procs]
        pm = [p['mem']  for p in procs]
        y  = np.arange(len(pn))
        ax4.barh(y-0.2, pc, 0.35, label='CPU%', color='#4f7cff', alpha=0.85)
        ax4.barh(y+0.2, pm, 0.35, label='MEM%', color='#22c55e', alpha=0.85)
        ax4.set_yticks(y); ax4.set_yticklabels(pn, fontsize=8)
        ax4.set_title('Top 10 Processes'); ax4.legend(fontsize=8)

    # Connections
    ax5 = fig.add_subplot(gs[1, 2])
    conns_all = psutil.net_connections(kind='inet')
    sc = Counter(c.status for c in conns_all)
    if sc:
        ax5.bar(list(sc.keys()), list(sc.values()),
                color=[PAL[i%len(PAL)] for i in range(len(sc))],
                edgecolor='white')
        ax5.set_title(f'Connections ({len(conns_all)} total)')
        ax5.tick_params(axis='x', rotation=45)

    # Agent votes
    ax6 = fig.add_subplot(gs[1, 3])
    agents  = STATUS['summ'].get('agents', {})
    if agents:
        ag_n = list(agents.keys())
        ag_c = [agents[n]['conf'] for n in ag_n]
        ag_v = [agents[n]['verdict'] for n in ag_n]
        ag_col = ['#ef4444' if vv not in ('normal','NORMAL',
                   'Benign','Normal','cluster_0') else '#22c55e'
                   for vv in ag_v]
        ax6.barh(ag_n[::-1], ag_c[::-1], color=ag_col[::-1], edgecolor='white')
        ax6.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
        ax6.set_xlim(0,1)
        for i,(c,vv) in enumerate(zip(ag_c[::-1],ag_v[::-1])):
            ax6.text(c+0.01, i, vv[:14], va='center', fontsize=6)
    ax6.set_title('Agent Votes')

    # Threat gauge
    ax7 = fig.add_subplot(gs[2, :2])
    s     = STATUS['score']
    theta = np.linspace(0, np.pi, 300)
    for lo, hi, col in [(0,0.1,'#22c55e'),(0.1,0.3,'#06b6d4'),
                         (0.3,0.5,'#f59e0b'),(0.5,0.7,'#f97316'),(0.7,1,'#ef4444')]:
        t  = theta[(theta>=lo*np.pi)&(theta<=hi*np.pi)]
        if len(t)<2: continue
        xo = np.cos(t); yo = np.sin(t)
        xi = np.cos(t)*0.6; yi = np.sin(t)*0.6
        ax7.fill(np.concatenate([xo,xi[::-1]]),
                  np.concatenate([yo,yi[::-1]]),color=col,alpha=0.85)
    needle = s * np.pi
    ax7.annotate('', xy=(np.cos(needle)*0.85, np.sin(needle)*0.85),
                  xytext=(0,0),
                  arrowprops=dict(arrowstyle='->', color='black', lw=2.5))
    ax7.text(0,-0.12,f'{s:.3f}',ha='center',fontsize=22,fontweight='bold')
    ax7.text(0,-0.30,STATUS['severity'],ha='center',fontsize=13,
              fontweight='bold',
              color=SEV_COL.get(STATUS['severity'],'gray'))
    ax7.text(0,-0.46,f'Family: {STATUS["summ"].get("attack_family","?")}',
              ha='center',fontsize=10,color='#94a3b8')
    ax7.set_xlim(-1.2,1.2); ax7.set_ylim(-0.55,1.2); ax7.axis('off')
    ax7.set_title('Threat Score Gauge')

    # Trend history
    ax8 = fig.add_subplot(gs[2, 2:])
    if len(scan_history) > 2:
        xs   = range(len(scan_history))
        sc_v = [e['threat_score'] for e in scan_history]
        cp_v = [e['cpu']          for e in scan_history]
        mm_v = [e['mem']          for e in scan_history]
        ax8.fill_between(xs, sc_v, alpha=0.25, color='#ef4444')
        ax8.plot(xs, sc_v, color='#ef4444', lw=1.5, label='Threat')
        ax8.plot(xs, cp_v, color='#4f7cff', lw=1.0, alpha=0.6, label='CPU%')
        ax8.plot(xs, mm_v, color='#22c55e', lw=1.0, alpha=0.6, label='MEM%')
        ax8.axhline(ALERT_THRESH, color='orange', linestyle='--',
                     alpha=0.5, label=f'Alert {ALERT_THRESH}')
        ax8.set_ylim(0,100); ax8.legend(fontsize=7)
        ax8.set_title('Trend History')
    else:
        ax8.text(0.5,0.5,'Collecting history...',
                  ha='center',va='center',transform=ax8.transAxes)
        ax8.set_title('Trend History')

    plt.tight_layout()
    if save:
        save_p = os.path.join(BASE, 'reports',
            f'dashboard_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png')
        plt.savefig(save_p, dpi=100, bbox_inches='tight')
        write_log(f'Dashboard saved: {save_p}')
    plt.show()


def plot_network_graph():
    if not HAS_NX:
        print('networkx not installed. pip install networkx')
        return
    procs     = get_process_snapshot()[:35]
    conns     = get_network_snapshot()
    G         = nx.DiGraph()

    for p in procs:
        col = ('#ef4444' if p['cpu']>50 else
               '#f59e0b' if p['cpu']>20 else '#4f7cff')
        G.add_node(f"P:{p['name']}:{p['pid']}",
                   label=p['name'], color=col,
                   size=max(300, p['cpu']*25))

    for c in conns:
        pid_node = next(
            (f"P:{p['name']}:{p['pid']}" for p in procs if p['pid']==c['pid']),
            None)
        if not pid_node or pid_node not in G.nodes: continue
        ip_node = f"IP:{c['rip']}"
        if ip_node not in G.nodes:
            G.add_node(ip_node,
                        label=c['rip'],
                        color='#ef4444' if c['malicious'] else '#94a3b8',
                        size=300)
        G.add_edge(pid_node, ip_node, port=c['rport'])

    if not G.nodes:
        print('No graph data.'); return

    fig, ax = plt.subplots(figsize=(18, 11))
    fig.patch.set_facecolor('#0a0d14')
    ax.set_facecolor('#0a0d14')

    try:    pos = nx.kamada_kawai_layout(G)
    except: pos = nx.spring_layout(G, seed=RANDOM, k=1.5)

    node_colors = [G.nodes[n].get('color','#4f7cff') for n in G.nodes]
    node_sizes  = [G.nodes[n].get('size',300)         for n in G.nodes]
    labels      = {n: G.nodes[n].get('label',n)[:14]  for n in G.nodes}

    nx.draw_networkx_nodes(G, pos, ax=ax,
        node_color=node_colors, node_size=node_sizes, alpha=0.85)
    nx.draw_networkx_edges(G, pos, ax=ax,
        edge_color='#475569', arrows=True, arrowsize=12,
        width=0.8, alpha=0.5, connectionstyle='arc3,rad=0.1')
    nx.draw_networkx_labels(G, pos, labels, ax=ax,
        font_size=7, font_color='white')

    patches = [
        mpatches.Patch(color='#4f7cff', label='Normal process'),
        mpatches.Patch(color='#f59e0b', label='High CPU >20%'),
        mpatches.Patch(color='#ef4444', label='Very high CPU / Malicious IP'),
        mpatches.Patch(color='#94a3b8', label='Remote IP'),
    ]
    ax.legend(handles=patches, loc='upper left',
              facecolor='#1e293b', labelcolor='white', fontsize=8)
    ax.set_title(
        f'Process-Connection Graph  [{datetime.now().strftime("%H:%M:%S")}]'
        f'  |  {len(procs)} processes  |  {len(conns)} connections',
        color='white', fontsize=12, fontweight='bold')
    ax.axis('off')

    save_p = os.path.join(BASE,'graphs',
        f'graph_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png')
    plt.savefig(save_p, dpi=100, bbox_inches='tight', facecolor='#0a0d14')
    plt.tight_layout()
    plt.show()
    write_log(f'Graph saved: {save_p}')


def plot_shap_dashboard():
    if not HAS_SHAP or SHAP_EXPLN is None:
        print('SHAP not available. Run NB5 first.')
        return
    feat = STATUS.get('feat')
    if feat is None:
        feat = extract_features()
    fi = explain_with_shap(feat, top_n=12)
    if fi:
        print('SHAP explanation complete.')
        for k, v in list(fi.items())[:10]:
            bar = '█' * int(abs(v)*30)
            print(f'  {k[:30]:<30} {v:+.4f}  {bar}')


def run_va_scan():
    DANGEROUS_PORTS = {
        21:'FTP',22:'SSH',23:'Telnet',25:'SMTP',
        135:'MS-RPC',139:'NetBIOS',445:'SMB',
        3389:'RDP',4444:'Meterpreter',5900:'VNC',
        6379:'Redis',27017:'MongoDB',1433:'MSSQL',
    }
    conns   = psutil.net_connections(kind='inet')
    exposed = []
    score   = 100
    for c in conns:
        if c.status == 'LISTEN' and c.laddr:
            if c.laddr.ip in ('0.0.0.0','::'):
                svc = DANGEROUS_PORTS.get(c.laddr.port,'Unknown')
                exposed.append(f'Port {c.laddr.port} ({svc})')
                score -= 10

    SUSP = {'mimikatz','meterpreter','nmap','netcat','nc','psexec','cobaltstrike'}
    bad_procs = []
    for p in psutil.process_iter(['name','pid']):
        try:
            if (p.info['name'] or '').lower() in SUSP:
                bad_procs.append(f"{p.info['name']} (pid={p.info['pid']})")
                score -= 15
        except: pass
    score = max(0, score)

    print('='*55)
    print('  VULNERABILITY ASSESSMENT')
    print('='*55)
    print(f'  Security Score  : {score}/100')
    print(f'  Open ports      : {len(conns)}')
    print(f'  Exposed services: {len(exposed)}')
    for s in exposed:   print(f'    !! {s}')
    print(f'  Suspicious procs: {len(bad_procs)}')
    for b in bad_procs: print(f'    !! {b}')
    print('='*55)

    save_p = os.path.join(BASE,'reports',
        f'va_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json')
    with open(save_p,'w',encoding='utf-8') as f:
        json.dump({'score':score,'exposed':exposed,
                   'bad_procs':bad_procs,
                   'time':datetime.now().isoformat()}, f, indent=2)
    write_log(f'VA scan: score={score}/100 saved: {save_p}')
    return score, exposed, bad_procs


def scan_file_static(filepath):
    result = {
        'file'   : os.path.basename(filepath),
        'path'   : filepath,
        'size_kb': round(os.path.getsize(filepath)/1024,2),
        'time'   : datetime.now().isoformat(),
        'md5'    : '', 'sha256': '',
        'verdict': 'CLEAN',
        'entropy': 0.0,
        'alerts' : [],
        'pe_info': {},
    }
    try:
        with open(filepath,'rb') as f: data = f.read()
        result['md5']    = hashlib.md5(data).hexdigest()
        result['sha256'] = hashlib.sha256(data).hexdigest()

        if result['md5'] in BAD_HASHES or result['sha256'] in BAD_HASHES:
            result['alerts'].append('KNOWN MALWARE HASH')
            result['verdict'] = 'MALICIOUS'

        if len(data)>0:
            cnt   = Counter(data); total = len(data)
            ent   = -sum((v/total)*np.log2(v/total)
                         for v in cnt.values() if v>0)
            result['entropy'] = round(ent,3)
            if ent > 7.2:
                result['alerts'].append(f'HIGH ENTROPY ({ent:.2f}) — packed?')

        try:
            import pefile
            pe = pefile.PE(filepath)
            imps = []
            SUSP_IMP = ['virtualalloc','writeprocessmemory',
                        'createremotethread','shellexecute',
                        'internetopen','urldownloadtofile']
            if hasattr(pe,'DIRECTORY_ENTRY_IMPORT'):
                for entry in pe.DIRECTORY_ENTRY_IMPORT:
                    dll = entry.dll.decode(errors='replace')
                    imps.append(dll)
                    for imp in entry.imports:
                        if imp.name:
                            fn = imp.name.decode(errors='replace').lower()
                            if any(s in fn for s in SUSP_IMP):
                                result['alerts'].append(f'SUSP IMPORT: {fn}')
            secs = []
            for s in pe.sections:
                sn  = s.Name.decode(errors='replace').strip('\x00')
                se  = s.get_entropy()
                secs.append({'name':sn,'entropy':round(se,2)})
                if se > 7.0:
                    result['alerts'].append(f'HIGH SECTION ENTROPY: {sn} ({se:.2f})')
            result['pe_info'] = {
                'imports' : imps[:15],
                'sections': secs,
                'compile_time': str(datetime.fromtimestamp(
                    pe.FILE_HEADER.TimeDateStamp))
            }
            pe.close()
        except: pass

        try:
            import yara
            yr = os.path.join(BASE,'yara_rules','protecter_rules.yar')
            if os.path.exists(yr):
                rules   = yara.compile(filepath=yr)
                matches = rules.match(filepath)
                for m in matches:
                    result['alerts'].append(f'YARA: {m.rule}')
        except: pass

        if result['alerts'] and result['verdict'] != 'MALICIOUS':
            result['verdict'] = ('SUSPICIOUS' if len(result['alerts'])>=2
                                 else 'WARNING')

    except Exception as e:
        result['alerts'].append(f'Scan error: {e}')

    print(f'File   : {result["file"]}')
    print(f'Verdict: {result["verdict"]}')
    print(f'Size   : {result["size_kb"]} KB')
    print(f'Entropy: {result["entropy"]}')
    print(f'MD5    : {result["md5"][:20]}...')
    print(f'Alerts ({len(result["alerts"])}):'
          + (''.join(f'\n  !! {a}' for a in result['alerts']) or '  None'))

    save_p = os.path.join(BASE,'reports',
        f'filescan_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json')
    with open(save_p,'w',encoding='utf-8') as f:
        json.dump(result, f, indent=2, default=str)
    return result


print('All dashboard + analysis functions ready.')
print()
print('Available functions:')
print('  plot_resource_dashboard() — full chart dashboard')
print('  plot_network_graph()      — NetworkX process topology')
print('  plot_shap_dashboard()     — SHAP explainability')
print('  run_va_scan()             — vulnerability assessment')
print('  scan_file_static(path)    — static file analysis')

All dashboard + analysis functions ready.

Available functions:
  plot_resource_dashboard() — full chart dashboard
  plot_network_graph()      — NetworkX process topology
  plot_shap_dashboard()     — SHAP explainability
  run_va_scan()             — vulnerability assessment
  scan_file_static(path)    — static file analysis


In [11]:
import os, sys

BASE        = r'C:\Users\varma\OneDrive\Desktop\newml'
TRAY_SCRIPT = os.path.join(BASE, 'tray.py')

LINES = [
"import os, sys, json, time, threading, warnings",
"import numpy as np",
"warnings.filterwarnings('ignore')",
"",
"BASE     = r'C:\\Users\\varma\\OneDrive\\Desktop\\newml'",
"ENV_PATH = os.path.join(BASE, '.env')",
"",
"def load_env(path):",
"    if not os.path.exists(path): return",
"    with open(path, encoding='utf-8') as f:",
"        for line in f:",
"            line = line.strip()",
"            if not line or line.startswith('#'): continue",
"            if '=' not in line: continue",
"            k, v = line.split('=', 1)",
"            os.environ[k.strip()] = v.strip()",
"",
"load_env(ENV_PATH)",
"",
"MODELS       = os.path.join(BASE, 'models')",
"NB1_MDL      = os.path.join(BASE, 'ml')",
"NB2_MDL      = os.path.join(BASE, 'dl')",
"NB3_MDL      = os.path.join(BASE, 'advanced')",
"LOG_FILE     = os.path.join(BASE, 'logs', 'protector_v2.log')",
"GROQ_KEY     = os.environ.get('GROQ_KEY', '')",
"GROQ_MODEL   = os.environ.get('GROQ_MODEL', 'llama-3.3-70b-versatile')",
"ALERT_THRESH = float(os.environ.get('THREAT_ALERT_THRESHOLD', '0.5'))",
"ENABLE_DL    = os.environ.get('ENABLE_DL_MODELS', 'true').lower() == 'true'",
"",
"import joblib, psutil, hashlib",
"from datetime import datetime",
"from collections import Counter, deque",
"from PIL import Image, ImageDraw",
"import pystray",
"from plyer import notification",
"",
"HISTORY       = deque(maxlen=200)",
"alert_log     = []",
"scan_history  = []",
"MALICIOUS_IPS = set()",
"STATUS        = {'verdict': 'Starting', 'severity': 'UNKNOWN',",
"                 'score': 0.0, 'summ': {}, 'feat': None}",
"",
"def write_log(msg, level='INFO'):",
"    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')",
"    try:",
"        with open(LOG_FILE, 'a', encoding='utf-8') as f:",
"            f.write(f'[{ts}] [{level}] {msg}\\n')",
"    except: pass",
"    print(f'[{ts}] [{level}] {msg}')",
"",
"def smart_load(fname):",
"    for folder in [MODELS, NB1_MDL, NB2_MDL, NB3_MDL]:",
"        path = os.path.join(folder, fname)",
"        if not os.path.exists(path): continue",
"        try:",
"            if fname.endswith('.pkl'):   return joblib.load(path)",
"            if fname.endswith('.json'):",
"                with open(path, encoding='utf-8') as f: return json.load(f)",
"            if fname.endswith('.txt'):",
"                with open(path, encoding='utf-8') as f:",
"                    s = set(f.read().splitlines())",
"                    s.discard('')",
"                    return s",
"            if fname.endswith('.npy'):   return np.load(path, allow_pickle=True)",
"        except: pass",
"    return None",
"",
"write_log('Loading models...')",
"SCALER      = smart_load('v2_scaler.pkl')",
"CLASSES     = smart_load('v2_label_classes.json')",
"OCSVM       = smart_load('v2_ocsvm.pkl')",
"ISO_FOREST  = smart_load('v2_isoforest.pkl')",
"SVM_BIN     = smart_load('v2_svm.pkl')",
"DT_MULTI    = smart_load('v2_dt.pkl')",
"RF_MULTI    = smart_load('v2_rf.pkl')",
"GNB_MULTI   = smart_load('v2_gnb.pkl')",
"ELM_MULTI   = smart_load('v2_elm.pkl')",
"GBC_ENS     = smart_load('v2_gbc.pkl')",
"ADA_ENS     = smart_load('v2_ada.pkl')",
"BAG_ENS     = smart_load('v2_bagging.pkl')",
"VOTING_ENS  = smart_load('v2_voting.pkl')",
"HMM_MODELS  = smart_load('v2_hmm_models.pkl')",
"KMEANS      = smart_load('v2_kmeans.pkl')",
"FAMILY_CLF  = smart_load('v2_family_clf.pkl')",
"FAMILY_ENC  = smart_load('v2_family_encoder.pkl')",
"FAMILY_NAMES= smart_load('v2_family_names.json')",
"META_CLF    = smart_load('v2_ensemble_meta.pkl')",
"META_ORDER  = smart_load('v2_ensemble_order.json')",
"FEAT_NAMES  = smart_load('v2_feature_names.json')",
"BAD_HASHES  = smart_load('bad_hashes.txt') or set()",
"TI          = smart_load('threat_intel.json') or {}",
"MALICIOUS_IPS = set(TI.get('malicious_ips', []))",
"",
"HAS_DL = False",
"DNN_MDL = AE_MDL = None",
"AE_THRESH = 0.01",
"NB2_CLASSES = CLASSES",
"",
"if ENABLE_DL:",
"    try:",
"        import tensorflow as tf",
"        def lk(fname):",
"            for folder in [MODELS, NB2_MDL]:",
"                p = os.path.join(folder, fname)",
"                if os.path.exists(p): return tf.keras.models.load_model(p)",
"            return None",
"        DNN_MDL     = lk('v2_dnn.keras')",
"        AE_MDL      = lk('v2_autoencoder.keras')",
"        ae_t        = smart_load('v2_ae_threshold.json')",
"        AE_THRESH   = ae_t.get('threshold', 0.01) if ae_t else 0.01",
"        NB2_CLASSES = smart_load('v2_label_classes.json') or CLASSES",
"        HAS_DL      = True",
"        write_log('DL models loaded.')",
"    except Exception as e:",
"        write_log(f'DL unavailable: {e}', 'WARN')",
"",
"write_log('All models loaded.')",
"",
"_LM = np.array([10,30,50,500,1e8,2e8,0,50,2e6,0.5,0,8000], dtype=np.float32)",
"_LS = np.array([15,20,80,300,3e8,5e8,1,100,5e6,0.3,0.5,20000], dtype=np.float32)",
"",
"def scale_live(f):",
"    return ((f - _LM) / (_LS + 1e-9)).reshape(1, -1)",
"",
"def pad_to_78(f):",
"    v = np.zeros(78, dtype=np.float32)",
"    v[0]=f[11]; v[2]=f[4]+f[5]; v[3]=f[2]; v[4]=f[2]",
"    v[5]=f[4];  v[6]=f[5]",
"    v[14]=(f[4]+f[5])/max(1.0,f[2])",
"    v[15]=f[2]; v[41]=f[8]; v[53]=f[8]; v[75]=f[9]",
"    return v.reshape(1, -1)",
"",
"def scale_78(f):",
"    v78 = pad_to_78(f)",
"    if SCALER:",
"        try: return SCALER.transform(v78)",
"        except: pass",
"    return v78",
"",
"def extract_features():",
"    try:",
"        net   = psutil.net_io_counters()",
"        bsent = float(net.bytes_sent)",
"        brecv = float(net.bytes_recv)",
"        disk  = float(psutil.disk_io_counters().read_bytes +",
"                      psutil.disk_io_counters().write_bytes) / 1e9",
"    except:",
"        bsent = brecv = disk = 0.0",
"    cpu   = float(psutil.cpu_percent(interval=0.1))",
"    mem   = float(psutil.virtual_memory().percent)",
"    conns = float(len(psutil.net_connections(kind='inet')))",
"    thrs  = 0.0",
"    try:",
"        thrs = float(sum(p.num_threads()",
"                         for p in psutil.process_iter(['num_threads'])",
"                         if p.info['num_threads']))",
"    except: pass",
"    feat = np.array([cpu, mem, conns, thrs, bsent, brecv,",
"                     0.0, disk, bsent/max(1.0,conns),",
"                     min(1.0,conns/100.0), 0.0, 80.0],",
"                    dtype=np.float32)",
"    HISTORY.append({'time':datetime.now(),'cpu':cpu,'mem':mem,'conns':conns})",
"    return feat",
"",
"class AgentResult:",
"    def __init__(self, name, verdict, confidence, detail=''):",
"        self.name       = name",
"        self.verdict    = verdict",
"        self.confidence = confidence",
"        self.detail     = detail",
"        self.is_threat  = verdict not in ('normal','NORMAL','Benign',",
"                                           'Normal','cluster_0',0)",
"",
"def run_all_agents(feat_raw):",
"    results = []",
"    Xs12 = scale_live(feat_raw)",
"    X78  = pad_to_78(feat_raw)",
"    Xs78 = scale_78(feat_raw)",
"    for model, name, inp, kind in [",
"        (OCSVM,      'OC-SVM',    Xs12, 'anomaly'),",
"        (ISO_FOREST, 'IsoForest', Xs12, 'anomaly'),",
"        (SVM_BIN,    'SVM-Bin',   Xs12, 'binary'),",
"    ]:",
"        if model is None: continue",
"        try:",
"            if kind == 'anomaly':",
"                pred  = model.predict(inp)[0]",
"                score = model.decision_function(inp)[0]",
"                conf  = max(0.0, min(1.0, (score+1)/2))",
"                results.append(AgentResult(name,",
"                    'normal' if pred==1 else 'ANOMALY',",
"                    1-conf if pred==-1 else conf))",
"            else:",
"                pred  = model.predict(inp)[0]",
"                proba = model.predict_proba(inp)[0]",
"                results.append(AgentResult(name,",
"                    'ATTACK' if pred==1 else 'normal',",
"                    float(proba.max())))",
"        except: pass",
"    for name, model, use_sc in [",
"        ('DT',        DT_MULTI,   False),",
"        ('RF',        RF_MULTI,   False),",
"        ('GNB',       GNB_MULTI,  True),",
"        ('ELM',       ELM_MULTI,  True),",
"        ('GradBoost', GBC_ENS,    False),",
"        ('AdaBoost',  ADA_ENS,    False),",
"        ('Bagging',   BAG_ENS,    False),",
"        ('Voting',    VOTING_ENS, True),",
"    ]:",
"        if model is None or CLASSES is None: continue",
"        try:",
"            inp   = Xs78 if use_sc else X78",
"            pred  = model.predict(inp)[0]",
"            proba = model.predict_proba(inp)[0]",
"            cls   = CLASSES[pred] if pred < len(CLASSES) else 'unknown'",
"            results.append(AgentResult(name, cls, float(proba.max())))",
"        except: pass",
"    if META_CLF and META_ORDER and CLASSES:",
"        try:",
"            mp = []",
"            mm = {'RF':RF_MULTI,'DT':DT_MULTI,'GNB':GNB_MULTI,",
"                  'ELM':ELM_MULTI,'GBC':GBC_ENS}",
"            for mn in META_ORDER:",
"                m = mm.get(mn)",
"                if m is None: continue",
"                inp = Xs78 if mn in ('GNB','ELM') else X78",
"                mp.append(m.predict_proba(inp)[0])",
"            if mp:",
"                mf     = np.hstack(mp).reshape(1,-1)",
"                mpred  = META_CLF.predict(mf)[0]",
"                mproba = META_CLF.predict_proba(mf)[0]",
"                cls    = CLASSES[mpred] if mpred<len(CLASSES) else 'unknown'",
"                results.append(AgentResult('MetaEnsemble',cls,",
"                    float(mproba.max()),'stacked'))",
"        except: pass",
"    if FAMILY_CLF and FAMILY_NAMES:",
"        try:",
"            pf  = FAMILY_CLF.predict(X78)[0]",
"            prf = FAMILY_CLF.predict_proba(X78)[0]",
"            fam = FAMILY_NAMES[pf] if pf<len(FAMILY_NAMES) else 'unknown'",
"            results.append(AgentResult('FamilyCLF',fam,",
"                float(prf.max()),f'family:{fam}'))",
"        except: pass",
"    if KMEANS:",
"        try:",
"            c = KMEANS.predict(Xs12)[0]",
"            d = KMEANS.transform(Xs12)[0].min()",
"            results.append(AgentResult('KMeans',f'cluster_{c}',",
"                max(0.0,1.0-d/10.0),f'dist={d:.2f}'))",
"        except: pass",
"    if HMM_MODELS:",
"        try:",
"            chunk  = np.tile(Xs12[0],(20,1))",
"            scores = {}",
"            for cls, m in HMM_MODELS.items():",
"                try:    scores[cls] = m.score(chunk)/20",
"                except: scores[cls] = -1e9",
"            best = max(scores, key=scores.get)",
"            conf = min(1.0, max(0.0,(scores[best]+50)/100))",
"            results.append(AgentResult('HMM',best,conf,",
"                f'log-L={scores[best]:.2f}'))",
"        except: pass",
"    if HAS_DL and DNN_MDL and NB2_CLASSES:",
"        try:",
"            proba = DNN_MDL.predict(Xs78,verbose=0)[0]",
"            pred  = int(np.argmax(proba))",
"            cls   = NB2_CLASSES[pred] if pred<len(NB2_CLASSES) else 'unknown'",
"            results.append(AgentResult('DNN',cls,float(proba.max())))",
"        except: pass",
"    if HAS_DL and AE_MDL:",
"        try:",
"            T    = 20",
"            ae_f = AE_MDL.input_shape[-1]",
"            base = Xs12 if ae_f==12 else Xs78",
"            seq  = np.tile(base,(T,1)).reshape(1,T,-1)",
"            recon= AE_MDL.predict(seq,verbose=0)",
"            err  = float(np.mean((recon-seq)**2))",
"            is_a = err > AE_THRESH",
"            results.append(AgentResult('Autoencoder',",
"                'ANOMALY' if is_a else 'normal',",
"                min(1.0,err/AE_THRESH) if is_a else max(0.0,1.0-err/AE_THRESH),",
"                f'err={err:.5f}'))",
"        except: pass",
"    return results",
"",
"def fuse_agents(results):",
"    if not results: return 'unknown', 0.0, {}",
"    ta  = [r for r in results if r.is_threat]",
"    ts  = sum(r.confidence for r in ta) / max(len(results),1)",
"    nn  = [r.verdict for r in ta",
"           if r.verdict not in ('normal','NORMAL','ANOMALY',",
"                                 'Benign','Normal','cluster_0')]",
"    at  = Counter(nn).most_common(1)[0][0] if nn else \\",
"          ('ANOMALY' if ta else 'normal')",
"    fv  = [r.verdict for r in results if r.name=='FamilyCLF']",
"    fam = fv[0] if fv else 'Unknown'",
"    if   ts > 0.7: sev = 'CRITICAL'",
"    elif ts > 0.5: sev = 'HIGH'",
"    elif ts > 0.3: sev = 'MEDIUM'",
"    elif ts > 0.1: sev = 'LOW'",
"    else:          sev = 'CLEAN'",
"    return at, ts, {",
"        'verdict'      : at if ts > 0.2 else 'normal',",
"        'severity'     : sev,",
"        'threat_score' : round(ts,3),",
"        'attack_family': fam,",
"        'n_agents'     : len(results),",
"        'n_flagging'   : len(ta),",
"        'agents'       : {r.name: {'verdict':r.verdict,",
"                                    'conf':round(r.confidence,3),",
"                                    'detail':r.detail}",
"                          for r in results}",
"    }",
"",
"def scan_cycle():",
"    feat = extract_features()",
"    res  = run_all_agents(feat)",
"    v, s, summ = fuse_agents(res)",
"    STATUS.update({'verdict':v,'severity':summ['severity'],",
"                   'score':s,'summ':summ,'feat':feat})",
"    entry = {",
"        'time'         : datetime.now().isoformat(),",
"        'verdict'      : summ['verdict'],",
"        'severity'     : summ['severity'],",
"        'threat_score' : s,",
"        'attack_family': summ.get('attack_family','?'),",
"        'cpu'          : float(feat[0]),",
"        'mem'          : float(feat[1]),",
"        'conns'        : float(feat[2]),",
"    }",
"    scan_history.append(entry)",
"    if len(scan_history) > 500: scan_history.pop(0)",
"    if s > ALERT_THRESH:",
"        alert_log.append(entry)",
"        write_log(f'THREAT: {v} family={summ.get(\"attack_family\",\"?\")} '",
"                  f'score={s:.3f}', 'ALERT')",
"        try:",
"            notification.notify(",
"                title   = f'Protector v2 - {summ[\"severity\"]}',",
"                message = f'{v} ({summ.get(\"attack_family\",\"?\")}) '",
"                          f'| score={s:.2f}',",
"                timeout = 8)",
"        except: pass",
"    return entry, summ",
"",
"def make_icon(severity='CLEAN'):",
"    cm = {",
"        'CLEAN'   : ((20,3,3),  (120,35,35)),",
"        'LOW'     : ((20,10,3), (180,120,30)),",
"        'MEDIUM'  : ((20,15,3), (200,160,40)),",
"        'HIGH'    : ((30,10,3), (220,100,30)),",
"        'CRITICAL': ((60,8,8),  (226,75,74)),",
"    }",
"    fill, col = cm.get(severity, cm['CLEAN'])",
"    img = Image.new('RGBA', (64,64), (0,0,0,0))",
"    d   = ImageDraw.Draw(img)",
"    cx = cy = 32; r = 28; ir = r//2",
"    d.polygon([(cx,cy-r),(cx+r,cy),(cx,cy+r),(cx-r,cy)],",
"               outline=col, fill=fill)",
"    d.polygon([(cx,cy-ir),(cx+ir,cy),(cx,cy+ir),(cx-ir,cy)],",
"               outline=col, fill=(10,2,2))",
"    d.ellipse([(cx-3,cy-3),(cx+3,cy+3)], fill=col)",
"    return img",
"",
"def show_popup(title, msg):",
"    import tkinter as tk",
"    from tkinter import scrolledtext",
"    def _build():",
"        win = tk.Tk()",
"        win.title(title)",
"        win.attributes('-topmost', True)",
"        win.lift()",
"        win.focus_force()",
"        win.geometry('580x440')",
"        txt = scrolledtext.ScrolledText(",
"            win, wrap=tk.WORD,",
"            font=('Consolas', 10),",
"            padx=8, pady=8)",
"        txt.insert(tk.END, msg)",
"        txt.config(state='disabled')",
"        txt.pack(fill='both', expand=True, padx=6, pady=6)",
"        def close():",
"            win.destroy()",
"        tk.Button(win, text='Close',",
"                   command=close,",
"                   bg='#4f7cff', fg='white',",
"                   font=('Segoe UI', 10, 'bold'),",
"                   width=18).pack(pady=6)",
"        win.bind('<Return>', lambda e: close())",
"        win.bind('<Escape>', lambda e: close())",
"        win.protocol('WM_DELETE_WINDOW', close)",
"        win.after(150, lambda: (win.lift(), win.focus_force()))",
"        win.mainloop()",
"    t = threading.Thread(target=_build, daemon=False)",
"    t.start()",
"    t.join(timeout=120)",
"",
"def ask_question_dialog(prompt='Enter your question:'):",
"    result = [None]",
"    def _build():",
"        import tkinter as tk",
"        win = tk.Tk()",
"        win.title('Protector v2 - Ask AI')",
"        win.attributes('-topmost', True)",
"        win.lift()",
"        win.focus_force()",
"        win.geometry('520x200')",
"        win.resizable(False, False)",
"        tk.Label(win, text=prompt,",
"                  font=('Segoe UI', 11),",
"                  wraplength=480,",
"                  justify='left',",
"                  padx=12, pady=12).pack(fill='x')",
"        var   = tk.StringVar()",
"        entry = tk.Entry(win, textvariable=var,",
"                          font=('Segoe UI', 12), width=50)",
"        entry.pack(padx=14, pady=4, ipady=4)",
"        btn_frame = tk.Frame(win)",
"        btn_frame.pack(pady=10)",
"        def on_ok(e=None):",
"            result[0] = var.get().strip()",
"            win.destroy()",
"        def on_cancel(e=None):",
"            result[0] = None",
"            win.destroy()",
"        tk.Button(btn_frame, text='Ask',",
"                   command=on_ok,",
"                   bg='#4f7cff', fg='white',",
"                   font=('Segoe UI', 10, 'bold'),",
"                   width=14).pack(side='left', padx=8)",
"        tk.Button(btn_frame, text='Cancel',",
"                   command=on_cancel,",
"                   font=('Segoe UI', 10),",
"                   width=14).pack(side='left', padx=8)",
"        win.bind('<Return>', on_ok)",
"        win.bind('<Escape>', on_cancel)",
"        win.protocol('WM_DELETE_WINDOW', on_cancel)",
"        win.after(200, lambda: (",
"            win.lift(),",
"            win.focus_force(),",
"            entry.focus_set(),",
"            entry.focus_force()",
"        ))",
"        win.mainloop()",
"    t = threading.Thread(target=_build, daemon=False)",
"    t.start()",
"    t.join(timeout=300)",
"    return result[0]",
"",
"def pick_file_dialog():",
"    result = [None]",
"    def _build():",
"        import tkinter as tk",
"        from tkinter import filedialog",
"        win = tk.Tk()",
"        win.withdraw()",
"        win.attributes('-topmost', True)",
"        win.lift()",
"        win.focus_force()",
"        win.update()",
"        fp = filedialog.askopenfilename(",
"            parent    = win,",
"            title     = 'Protector v2 - Select File to Scan',",
"            filetypes = [",
"                ('Executables and Scripts',",
"                 '*.exe;*.dll;*.bat;*.cmd;*.ps1;*.vbs;*.js;*.py;*.sh'),",
"                ('Documents',",
"                 '*.pdf;*.doc;*.docx;*.xls;*.xlsx'),",
"                ('Archives',",
"                 '*.zip;*.rar;*.7z;*.tar;*.gz'),",
"                ('All files', '*.*'),",
"            ]",
"        )",
"        win.destroy()",
"        result[0] = fp if fp else None",
"    t = threading.Thread(target=_build, daemon=False)",
"    t.start()",
"    t.join(timeout=120)",
"    return result[0]",
"",
"def show_dashboard(icon=None, item=None):",
"    try:",
"        import matplotlib",
"        matplotlib.use('TkAgg')",
"        import matplotlib.pyplot as plt",
"        import matplotlib.patches as mpatches",
"        PAL     = ['#4f7cff','#22c55e','#f59e0b','#ef4444','#8b5cf6','#06b6d4']",
"        SEV_COL = {'CLEAN':'#22c55e','LOW':'#06b6d4','MEDIUM':'#f59e0b',",
"                   'HIGH':'#f97316','CRITICAL':'#ef4444'}",
"        feat = extract_features()",
"        res  = run_all_agents(feat)",
"        v, s, summ = fuse_agents(res)",
"        STATUS.update({'verdict':v,'severity':summ['severity'],",
"                       'score':s,'summ':summ,'feat':feat})",
"        fig = plt.figure(figsize=(22,13))",
"        fig.suptitle(",
"            f\"Protector v2  [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]\"",
"            f\"   Verdict: {summ['verdict']}  |  Severity: {summ['severity']}\"",
"            f\"  |  Score: {s:.3f}  |  Family: {summ.get('attack_family','?')}\",",
"            fontsize=12, fontweight='bold', y=0.99)",
"        gs = fig.add_gridspec(3,4,hspace=0.45,wspace=0.35)",
"        ax1 = fig.add_subplot(gs[0,:2])",
"        cpu_cores = psutil.cpu_percent(percpu=True)",
"        cols_cpu  = ['#ef4444' if c>80 else '#f59e0b' if c>50",
"                     else '#4f7cff' for c in cpu_cores]",
"        ax1.bar(range(len(cpu_cores)),cpu_cores,color=cols_cpu,edgecolor='white')",
"        ax1.set_ylim(0,100)",
"        ax1.axhline(80,color='#ef4444',linestyle='--',alpha=0.5)",
"        ax1.set_title(f\"CPU Per Core  (avg={sum(cpu_cores)/len(cpu_cores):.1f}%)\")",
"        ax2 = fig.add_subplot(gs[0,2])",
"        mem = psutil.virtual_memory()",
"        ax2.pie([mem.used,mem.available],",
"                labels=[f'Used\\n{mem.used/1e9:.1f}GB',",
"                        f'Free\\n{mem.available/1e9:.1f}GB'],",
"                colors=['#ef4444','#22c55e'],",
"                autopct='%1.1f%%',startangle=90,",
"                textprops={'fontsize':8})",
"        ax2.set_title(f'RAM ({mem.total/1e9:.1f}GB)')",
"        ax3 = fig.add_subplot(gs[0,3])",
"        net = psutil.net_io_counters()",
"        ax3.bar(['Sent MB','Recv MB'],",
"                [net.bytes_sent/1e6,net.bytes_recv/1e6],",
"                color=[PAL[0],PAL[1]],edgecolor='white')",
"        ax3.set_title('Network I/O')",
"        ax4 = fig.add_subplot(gs[1,:2])",
"        procs = []",
"        for p in psutil.process_iter(['name','cpu_percent','memory_percent']):",
"            try:",
"                procs.append({",
"                    'name': (p.info.get('name') or '?')[:14],",
"                    'cpu' : p.info.get('cpu_percent')    or 0.0,",
"                    'mem' : p.info.get('memory_percent') or 0.0,",
"                })",
"            except: pass",
"        procs = sorted(procs,key=lambda x:x['cpu'],reverse=True)[:10]",
"        if procs:",
"            pn=[p['name'] for p in procs]",
"            pc=[p['cpu']  for p in procs]",
"            pm=[p['mem']  for p in procs]",
"            y=np.arange(len(pn))",
"            ax4.barh(y-0.2,pc,0.35,label='CPU%',color=PAL[0],alpha=0.85)",
"            ax4.barh(y+0.2,pm,0.35,label='MEM%',color=PAL[1],alpha=0.85)",
"            ax4.set_yticks(y); ax4.set_yticklabels(pn,fontsize=8)",
"            ax4.set_title('Top 10 Processes'); ax4.legend(fontsize=8)",
"        ax5 = fig.add_subplot(gs[1,2])",
"        conns_all = psutil.net_connections(kind='inet')",
"        sc = Counter(c.status for c in conns_all)",
"        if sc:",
"            ax5.bar(list(sc.keys()),list(sc.values()),",
"                    color=[PAL[i%len(PAL)] for i in range(len(sc))],",
"                    edgecolor='white')",
"            ax5.set_title(f'Connections ({len(conns_all)} total)')",
"            ax5.tick_params(axis='x',rotation=45)",
"        ax6 = fig.add_subplot(gs[1,3])",
"        agents = summ.get('agents',{})",
"        if agents:",
"            an  = list(agents.keys())",
"            ac  = [agents[n]['conf']    for n in an]",
"            av  = [agents[n]['verdict'] for n in an]",
"            acol= ['#ef4444' if vv not in",
"                   ('normal','NORMAL','Benign','Normal','cluster_0')",
"                   else '#22c55e' for vv in av]",
"            ax6.barh(an[::-1],ac[::-1],color=acol[::-1],edgecolor='white')",
"            ax6.axvline(0.5,color='gray',linestyle='--',alpha=0.5)",
"            ax6.set_xlim(0,1)",
"            for i,(c,vv) in enumerate(zip(ac[::-1],av[::-1])):",
"                ax6.text(c+0.01,i,vv[:14],va='center',fontsize=6)",
"        ax6.set_title('Agent Votes')",
"        ax7 = fig.add_subplot(gs[2,:2])",
"        theta = np.linspace(0,np.pi,300)",
"        for lo,hi,col in [",
"            (0.0,0.1,'#22c55e'),(0.1,0.3,'#06b6d4'),",
"            (0.3,0.5,'#f59e0b'),(0.5,0.7,'#f97316'),(0.7,1.0,'#ef4444')]:",
"            t=theta[(theta>=lo*np.pi)&(theta<=hi*np.pi)]",
"            if len(t)<2: continue",
"            xo=np.cos(t); yo=np.sin(t)",
"            xi=np.cos(t)*0.6; yi=np.sin(t)*0.6",
"            ax7.fill(np.concatenate([xo,xi[::-1]]),",
"                      np.concatenate([yo,yi[::-1]]),",
"                      color=col,alpha=0.85)",
"        needle = s*np.pi",
"        ax7.annotate('',",
"                      xy=(np.cos(needle)*0.85,np.sin(needle)*0.85),",
"                      xytext=(0,0),",
"                      arrowprops=dict(arrowstyle='->',color='black',lw=2.5))",
"        ax7.text(0,-0.12,f'{s:.3f}',ha='center',fontsize=22,fontweight='bold')",
"        ax7.text(0,-0.30,summ['severity'],ha='center',fontsize=13,",
"                  fontweight='bold',",
"                  color=SEV_COL.get(summ['severity'],'gray'))",
"        ax7.text(0,-0.46,",
"                  f\"Family: {summ.get('attack_family','?')}\",",
"                  ha='center',fontsize=10,color='#94a3b8')",
"        ax7.set_xlim(-1.2,1.2); ax7.set_ylim(-0.55,1.2); ax7.axis('off')",
"        ax7.set_title('Threat Score Gauge')",
"        ax8 = fig.add_subplot(gs[2,2:])",
"        if len(scan_history) > 2:",
"            xs   = range(len(scan_history))",
"            sc_v = [e['threat_score'] for e in scan_history]",
"            cp_v = [e['cpu']          for e in scan_history]",
"            mm_v = [e['mem']          for e in scan_history]",
"            ax8.fill_between(xs,sc_v,alpha=0.25,color='#ef4444')",
"            ax8.plot(xs,sc_v,color='#ef4444',lw=1.5,label='Threat')",
"            ax8.plot(xs,cp_v,color='#4f7cff',lw=1.0,alpha=0.6,label='CPU%')",
"            ax8.plot(xs,mm_v,color='#22c55e',lw=1.0,alpha=0.6,label='MEM%')",
"            ax8.axhline(ALERT_THRESH,color='orange',linestyle='--',alpha=0.5)",
"            ax8.set_ylim(0,100); ax8.legend(fontsize=7)",
"        ax8.set_title('Trend History')",
"        plt.tight_layout()",
"        save_p = os.path.join(BASE,'reports',",
"            f\"dashboard_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png\")",
"        os.makedirs(os.path.dirname(save_p),exist_ok=True)",
"        plt.savefig(save_p,dpi=100,bbox_inches='tight')",
"        plt.show()",
"        write_log(f'Dashboard saved: {save_p}')",
"    except Exception as e:",
"        write_log(f'Dashboard error: {e}','WARN')",
"        show_popup('Dashboard Error', str(e))",
"",
"def show_network_graph(icon=None, item=None):",
"    try:",
"        import networkx as nx",
"        import matplotlib",
"        matplotlib.use('TkAgg')",
"        import matplotlib.pyplot as plt",
"        import matplotlib.patches as mpatches",
"        procs = []",
"        for p in psutil.process_iter(['pid','name','cpu_percent',",
"                                       'memory_percent','connections']):",
"            try:",
"                procs.append({'pid':p.info['pid'],",
"                               'name':(p.info['name'] or '?')[:15],",
"                               'cpu':p.info['cpu_percent'] or 0.0,",
"                               'mem':p.info['memory_percent'] or 0.0,",
"                               'conns':p.info['connections'] or []})",
"            except: pass",
"        procs = sorted(procs,key=lambda x:x['cpu'],reverse=True)[:30]",
"        conns_all = []",
"        for c in psutil.net_connections(kind='inet'):",
"            if c.raddr:",
"                conns_all.append({'rip':c.raddr.ip,'rport':c.raddr.port,",
"                                   'pid':c.pid or 0,",
"                                   'malicious':c.raddr.ip in MALICIOUS_IPS})",
"        G = nx.DiGraph()",
"        for p in procs:",
"            col = ('#ef4444' if p['cpu']>50 else",
"                   '#f59e0b' if p['cpu']>20 else '#4f7cff')",
"            G.add_node(f\"P:{p['name']}:{p['pid']}\",",
"                       label=p['name'],color=col,size=max(300,p['cpu']*25))",
"        for c in conns_all:",
"            pn = next((f\"P:{p['name']}:{p['pid']}\"",
"                       for p in procs if p['pid']==c['pid']),None)",
"            if not pn or pn not in G.nodes: continue",
"            ip_n = f\"IP:{c['rip']}\"",
"            if ip_n not in G.nodes:",
"                G.add_node(ip_n,label=c['rip'],",
"                            color='#ef4444' if c['malicious'] else '#94a3b8',",
"                            size=300)",
"            G.add_edge(pn,ip_n,port=c['rport'])",
"        if not G.nodes:",
"            show_popup('Process Graph','No data to display.')",
"            return",
"        fig,ax = plt.subplots(figsize=(18,11))",
"        fig.patch.set_facecolor('#0a0d14')",
"        ax.set_facecolor('#0a0d14')",
"        try:    pos = nx.kamada_kawai_layout(G)",
"        except: pos = nx.spring_layout(G,seed=42,k=1.5)",
"        nc = [G.nodes[n].get('color','#4f7cff') for n in G.nodes]",
"        ns = [G.nodes[n].get('size',300)         for n in G.nodes]",
"        lb = {n:G.nodes[n].get('label',n)[:14]   for n in G.nodes}",
"        nx.draw_networkx_nodes(G,pos,ax=ax,node_color=nc,node_size=ns,alpha=0.85)",
"        nx.draw_networkx_edges(G,pos,ax=ax,edge_color='#475569',arrows=True,",
"                               arrowsize=12,width=0.8,alpha=0.5,",
"                               connectionstyle='arc3,rad=0.1')",
"        nx.draw_networkx_labels(G,pos,lb,ax=ax,font_size=7,font_color='white')",
"        patches = [",
"            mpatches.Patch(color='#4f7cff',label='Normal process'),",
"            mpatches.Patch(color='#f59e0b',label='High CPU >20%'),",
"            mpatches.Patch(color='#ef4444',label='CPU >50% or Malicious IP'),",
"            mpatches.Patch(color='#94a3b8',label='Remote IP'),",
"        ]",
"        ax.legend(handles=patches,loc='upper left',",
"                   facecolor='#1e293b',labelcolor='white',fontsize=8)",
"        ax.set_title(",
"            f\"Process Graph [{datetime.now().strftime('%H:%M:%S')}]\"",
"            f\"  |  {len(procs)} procs  |  {len(conns_all)} conns\",",
"            color='white',fontsize=12,fontweight='bold')",
"        ax.axis('off')",
"        save_p = os.path.join(BASE,'graphs',",
"            f\"graph_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png\")",
"        os.makedirs(os.path.dirname(save_p),exist_ok=True)",
"        plt.savefig(save_p,dpi=100,bbox_inches='tight',facecolor='#0a0d14')",
"        plt.tight_layout(); plt.show()",
"        write_log(f'Graph saved: {save_p}')",
"    except Exception as e:",
"        write_log(f'Graph error: {e}','WARN')",
"        show_popup('Graph Error',str(e))",
"",
"def show_status(icon=None, item=None):",
"    s = STATUS",
"    agents_str = '\\n'.join(",
"        f\"  {k}: {v['verdict']} ({v['conf']:.2f})  {v.get('detail','')}\"",
"        for k,v in s.get('summ',{}).get('agents',{}).items()",
"    )",
"    msg = (",
"        f\"Verdict       : {s['verdict']}\\n\"",
"        f\"Severity      : {s['severity']}\\n\"",
"        f\"Score         : {s['score']:.3f}\\n\"",
"        f\"Attack Family : {s.get('summ',{}).get('attack_family','?')}\\n\"",
"        f\"Time          : {datetime.now().strftime('%H:%M:%S')}\\n\"",
"        f\"Agents firing : {s.get('summ',{}).get('n_flagging','?')}/\"",
"        f\"{s.get('summ',{}).get('n_agents','?')}\\n\\n\"",
"        f\"Agent Details:\\n{agents_str}\"",
"    )",
"    show_popup('Protector v2 - Status', msg)",
"",
"def show_va(icon=None, item=None):",
"    DANGEROUS_PORTS = {",
"        21:'FTP',22:'SSH',23:'Telnet',25:'SMTP',",
"        135:'MS-RPC',139:'NetBIOS',445:'SMB',",
"        3389:'RDP',4444:'Meterpreter',5900:'VNC',",
"    }",
"    conns   = psutil.net_connections(kind='inet')",
"    exposed = []",
"    score   = 100",
"    for c in conns:",
"        if c.status=='LISTEN' and c.laddr:",
"            if c.laddr.ip in ('0.0.0.0','::'):",
"                svc = DANGEROUS_PORTS.get(c.laddr.port,'Unknown')",
"                exposed.append(f'Port {c.laddr.port} ({svc})')",
"                score -= 10",
"    SUSP = {'mimikatz','meterpreter','nmap','netcat','nc','psexec'}",
"    bad_procs = []",
"    for p in psutil.process_iter(['name','pid']):",
"        try:",
"            if (p.info['name'] or '').lower() in SUSP:",
"                bad_procs.append(",
"                    f\"{p.info['name']} (pid={p.info['pid']})\")",
"                score -= 15",
"        except: pass",
"    score = max(0,score)",
"    msg = (",
"        f'Security Score : {score}/100\\n\\n'",
"        f'Exposed services ({len(exposed)}):\\n'",
"        + ('\\n'.join(f'  !! {e}' for e in exposed) or '  None')",
"        + f'\\n\\nSuspicious processes ({len(bad_procs)}):\\n'",
"        + ('\\n'.join(f'  !! {b}' for b in bad_procs) or '  None')",
"        + f'\\n\\nActive connections : {len(conns)}'",
"        + f'\\nMalicious IPs known: {len(MALICIOUS_IPS)}'",
"    )",
"    write_log(f'VA scan: score={score}/100')",
"    show_popup('Protector v2 - VA Scan', msg)",
"",
"def scan_file_menu(icon=None, item=None):",
"    fp = pick_file_dialog()",
"    if not fp: return",
"    try:",
"        with open(fp,'rb') as f: data = f.read()",
"        md5   = hashlib.md5(data).hexdigest()",
"        sha   = hashlib.sha256(data).hexdigest()",
"        known = md5 in BAD_HASHES or sha in BAD_HASHES",
"        size  = len(data)",
"        ent_val = 0.0",
"        if size > 0:",
"            cnt   = Counter(data)",
"            total = size",
"            ent_val = -sum((v/total)*np.log2(v/total)",
"                           for v in cnt.values() if v>0)",
"        verdict = ('MALICIOUS'  if known else",
"                   'SUSPICIOUS' if ent_val>7.2 else 'CLEAN')",
"        alerts = []",
"        if known:         alerts.append('KNOWN MALWARE HASH')",
"        if ent_val > 7.2: alerts.append(f'HIGH ENTROPY ({ent_val:.2f})')",
"        try:",
"            import pefile",
"            pe = pefile.PE(fp)",
"            SUSP_IMP = ['virtualalloc','writeprocessmemory',",
"                        'createremotethread','shellexecute']",
"            if hasattr(pe,'DIRECTORY_ENTRY_IMPORT'):",
"                for entry in pe.DIRECTORY_ENTRY_IMPORT:",
"                    for imp in entry.imports:",
"                        if imp.name:",
"                            fn = imp.name.decode(errors='replace').lower()",
"                            if any(s in fn for s in SUSP_IMP):",
"                                alerts.append(f'SUSP IMPORT: {fn}')",
"            for s in pe.sections:",
"                sn = s.Name.decode(errors='replace').strip('\\x00')",
"                se = s.get_entropy()",
"                if se > 7.0:",
"                    alerts.append(f'HIGH SECTION ENTROPY: {sn} ({se:.2f})')",
"            pe.close()",
"        except: pass",
"        try:",
"            import yara",
"            yr = os.path.join(BASE,'yara_rules','protecter_rules.yar')",
"            if os.path.exists(yr):",
"                rules   = yara.compile(filepath=yr)",
"                matches = rules.match(fp)",
"                for m in matches:",
"                    alerts.append(f'YARA: {m.rule}')",
"        except: pass",
"        if alerts and verdict != 'MALICIOUS':",
"            verdict = 'SUSPICIOUS' if len(alerts)>=2 else 'WARNING'",
"        msg = (",
"            f'File    : {os.path.basename(fp)}\\n'",
"            f'Size    : {size:,} bytes\\n'",
"            f'Verdict : {verdict}\\n'",
"            f'Entropy : {ent_val:.3f}\\n'",
"            f'MD5     : {md5}\\n'",
"            f'SHA256  : {sha[:32]}...\\n'",
"            f'\\nAlerts ({len(alerts)}):\\n'",
"            + ('\\n'.join(f'  !! {a}' for a in alerts) or '  None found')",
"        )",
"        write_log(f'File scan: {fp} verdict={verdict}')",
"        show_popup(f'File Scan - {verdict}', msg)",
"    except Exception as e:",
"        show_popup('Scan Error', str(e))",
"",
"def ask_ai(icon=None, item=None):",
"    q = ask_question_dialog(",
"        'Type your security question and press Enter or click Ask:')",
"    if not q: return",
"    try:",
"        from groq import Groq",
"        s   = STATUS",
"        ctx = (",
"            f\"verdict={s['verdict']} severity={s['severity']}\"",
"            f\" score={s['score']:.3f}\"",
"            f\" family={s.get('summ',{}).get('attack_family','?')}\\n\"",
"            f\"agents={json.dumps(s.get('summ',{}).get('agents',{}))}\"",
"        )",
"        sys_p = (",
"            'You are Protector v2, expert AI cybersecurity agent. '",
"            'Reference the agent verdicts. '",
"            'Rate severity CRITICAL/HIGH/MEDIUM/LOW/CLEAN. '",
"            'Give specific actionable advice. Be concise.'",
"        )",
"        client = Groq(api_key=GROQ_KEY)",
"        resp   = client.chat.completions.create(",
"            model    = GROQ_MODEL,",
"            messages = [",
"                {'role':'system','content':sys_p},",
"                {'role':'user',",
"                 'content':f'Context:\\n{ctx}\\n\\nQuestion: {q}'}",
"            ],",
"            max_tokens  = 700,",
"            temperature = 0.2",
"        )",
"        answer = resp.choices[0].message.content",
"        write_log(f'Groq Q&A: {q[:60]}...')",
"        show_popup(f'AI Answer - {q[:40]}...', answer)",
"    except Exception as e:",
"        write_log(f'Groq error: {e}','WARN')",
"        show_popup('Groq Error',",
"            f'Error: {e}\\n\\nCheck GROQ_KEY in .env:\\n{ENV_PATH}')",
"",
"def poll_loop(icon):",
"    while True:",
"        try:",
"            entry, summ = scan_cycle()",
"            sev = summ['severity']",
"            icon.icon  = make_icon(sev)",
"            icon.title = (",
"                f\"Protector v2  |  {sev}\"",
"                f\"  |  score={summ['threat_score']:.2f}\"",
"                f\"  |  family={summ.get('attack_family','?')}\"",
"                f\"  |  CPU={entry['cpu']:.0f}%\"",
"                f\"  MEM={entry['mem']:.0f}%\"",
"            )",
"        except Exception as e:",
"            write_log(f'Poll error: {e}','WARN')",
"        time.sleep(60)",
"",
"icon = pystray.Icon(",
"    name  = 'protector_v2',",
"    icon  = make_icon('CLEAN'),",
"    title = 'Protector v2 - Starting...',",
"    menu  = pystray.Menu(",
"        pystray.MenuItem('Full Dashboard',  show_dashboard,  default=True),",
"        pystray.MenuItem('Process Graphs',  show_network_graph),",
"        pystray.MenuItem('Show Status',     show_status),",
"        pystray.MenuItem('VA Scan',         show_va),",
"        pystray.MenuItem('Scan File',       scan_file_menu),",
"        pystray.MenuItem('Ask AI (Groq)',   ask_ai),",
"        pystray.MenuItem('Quit',            lambda i,it: i.stop()),",
"    )",
")",
"",
"threading.Thread(target=poll_loop,args=(icon,),daemon=True).start()",
"write_log('Tray agent started.')",
"icon.run()",
]

# Write file
with open(TRAY_SCRIPT, 'w', encoding='utf-8') as f:
    f.write('\n'.join(LINES))

# Verify zero non-ASCII
bad = []
content = '\n'.join(LINES)
for i, line in enumerate(LINES, 1):
    for ch in line:
        if ord(ch) > 127:
            bad.append(f'  Line {i}: U+{ord(ch):04X}  {line.strip()[:50]}')

if bad:
    print(f'WARNING: {len(bad)} non-ASCII chars found:')
    for b in bad[:10]: print(b)
else:
    print('tray.py written with zero non-ASCII characters.')

print(f'Lines : {len(LINES)}')
print(f'File  : {TRAY_SCRIPT}')

tray.py written with zero non-ASCII characters.
Lines : 891
File  : C:\Users\varma\OneDrive\Desktop\newml\tray.py


In [12]:
import os, sys, shutil, subprocess, time

BASE        = r'C:\Users\varma\OneDrive\Desktop\newml'
TRAY_SCRIPT = os.path.join(BASE, 'tray.py')

PYTHONW = os.path.join(os.path.dirname(sys.executable), 'pythonw.exe')
for candidate in [
    os.path.join(sys.prefix, 'pythonw.exe'),
    sys.executable.replace('python.exe', 'pythonw.exe'),
    r'C:\Users\varma\anaconda3\pythonw.exe',
    r'C:\ProgramData\anaconda3\pythonw.exe',
]:
    if os.path.exists(candidate):
        PYTHONW = candidate
        break

print(f'pythonw : {PYTHONW}')

VBS_PATH = os.path.join(BASE, 'launch_protector.vbs')
with open(VBS_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join([
        'Dim WshShell',
        'Set WshShell = CreateObject("WScript.Shell")',
        f'Dim pypath : pypath = "{PYTHONW}"',
        f'Dim script : script = "{TRAY_SCRIPT}"',
        'Dim cmd : cmd = Chr(34) & pypath & Chr(34) & " " & Chr(34) & script & Chr(34)',
        'WshShell.Run cmd, 0, False',
        'Set WshShell = Nothing',
    ]))
print(f'OK: {VBS_PATH}')

BAT_PATH = os.path.join(BASE, 'launch_protector.bat')
with open(BAT_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join([
        '@echo off',
        f'cd /d "{BASE}"',
        f'start "" "{PYTHONW}" "{TRAY_SCRIPT}"',
    ]))
print(f'OK: {BAT_PATH}')

STARTUP_DIR = os.path.join(os.environ.get('APPDATA',''),
    r'Microsoft\Windows\Start Menu\Programs\Startup')
STARTUP_VBS = os.path.join(STARTUP_DIR, 'launch_protector.vbs')
if os.path.exists(STARTUP_VBS):
    os.remove(STARTUP_VBS)
shutil.copy(VBS_PATH, STARTUP_VBS)
print(f'Startup updated: {STARTUP_VBS}')

os.system('taskkill /f /im pythonw.exe >nul 2>&1')
time.sleep(2)
print('Killed old tray.')

subprocess.Popen(['wscript', VBS_PATH])
time.sleep(4)
print()
print('Tray launched.')
print('LEFT-CLICK  -> Full Dashboard')
print('RIGHT-CLICK -> Full menu')

pythonw : C:\Users\varma\anaconda3\pythonw.exe
OK: C:\Users\varma\OneDrive\Desktop\newml\launch_protector.vbs
OK: C:\Users\varma\OneDrive\Desktop\newml\launch_protector.bat
Startup updated: C:\Users\varma\AppData\Roaming\Microsoft\Windows\Start Menu\Programs\Startup\launch_protector.vbs
Killed old tray.

Tray launched.
LEFT-CLICK  -> Full Dashboard
RIGHT-CLICK -> Full menu
